# War Room Discussion — LLM-as-Judge Benchmark

Reads all captured war room JSON files from `war_room_results/`, sends the discussion transcript to GPT-4o, and returns a score across 5 dimensions:

| Dimension | What it measures |
|---|---|
| **argument_quality** | Are claims backed by specific data/quotes? |
| **reasoning_diversity** | Do agents surface genuinely different perspectives? |
| **engagement_quality** | Do agents actually respond to each other? |
| **conflict_resolution** | Are disagreements surfaced and resolved? |
| **discussion_utility** | Does the debate produce insights beyond individual analyses? |

In [ ]:
import json
import os
import sys
from pathlib import Path

# Add backend to path so we can import the existing judge module
BACKEND_DIR = Path("__file__").resolve().parent.parent
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

CAPTURES_DIR = Path("__file__").resolve().parent / "war_room_results"

print(f"Backend dir : {BACKEND_DIR}")
print(f"Captures dir: {CAPTURES_DIR}")
print(f"Files found : {list(CAPTURES_DIR.glob('*.json'))}")

In [ ]:
# ── Set your OpenAI API key ──────────────────────────────────────────────────
# Option A: set it here directly (don't commit this!)
# OPENAI_API_KEY = "sk-..."

# Option B: read from environment (recommended)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not set. Set it as an env var or paste it above.")

print("API key loaded ✓")

In [ ]:
from eval.metrics.warroom_judge import evaluate_warroom_discussion

SCORE_DIMS = [
    "argument_quality",
    "reasoning_diversity",
    "engagement_quality",
    "conflict_resolution",
    "discussion_utility",
]

def load_capture(path: Path) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

print("Judge module loaded ✓")

In [ ]:
# ── Run judge on all TC files ────────────────────────────────────────────────
results = []

capture_files = sorted(CAPTURES_DIR.glob("*.json"))
print(f"Found {len(capture_files)} capture file(s)\n")

for path in capture_files:
    capture = load_capture(path)
    test_id = capture.get("test_id", path.stem)
    description = capture.get("description", "")
    num_messages = len(capture.get("discussion", []))
    num_rounds = max((m["round"] for m in capture.get("discussion", [])), default=0)

    print(f"Judging {test_id} — {description}")
    print(f"  {num_messages} messages across {num_rounds} rounds")

    score = evaluate_warroom_discussion(
        {"discussion": capture["discussion"]},
        OPENAI_API_KEY
    )

    if score.get("skipped"):
        print(f"  ⚠️  Skipped: {score.get('reason')}\n")
        results.append({"test_id": test_id, "description": description, "skipped": True, "reason": score.get("reason")})
        continue

    print(f"  Overall score: {score['overall_score']}/10")
    for dim in SCORE_DIMS:
        print(f"    {dim:<25} {score[dim]:>2}/10  — {score.get(dim + '_rationale', '')}")
    print(f"  Strength : {score.get('key_strength', '')}")
    print(f"  Weakness : {score.get('key_weakness', '')}\n")

    results.append({
        "test_id": test_id,
        "description": description,
        "num_messages": num_messages,
        "num_rounds": num_rounds,
        **{dim: score[dim] for dim in SCORE_DIMS},
        "overall_score": score["overall_score"],
        "key_strength": score.get("key_strength", ""),
        "key_weakness": score.get("key_weakness", ""),
    })

print("Done.")

In [ ]:
# ── Summary table ────────────────────────────────────────────────────────────
import pandas as pd

scored = [r for r in results if not r.get("skipped")]

if not scored:
    print("No scores to display.")
else:
    df = pd.DataFrame(scored).set_index("test_id")
    display_cols = SCORE_DIMS + ["overall_score"]
    print(df[["description", "num_messages", "num_rounds"] + display_cols].to_string())

In [ ]:
# ── Bar chart — scores per test case ────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

if scored:
    labels = [r["test_id"] for r in scored]
    x = np.arange(len(SCORE_DIMS))
    width = 0.8 / len(scored)

    fig, ax = plt.subplots(figsize=(12, 5))
    for i, r in enumerate(scored):
        vals = [r[dim] for dim in SCORE_DIMS]
        offset = (i - len(scored) / 2 + 0.5) * width
        bars = ax.bar(x + offset, vals, width, label=r["test_id"])

    ax.set_ylabel("Score (0–10)")
    ax.set_title("War Room Discussion — LLM-as-Judge Scores by Dimension")
    ax.set_xticks(x)
    ax.set_xticklabels([d.replace("_", "\n") for d in SCORE_DIMS], fontsize=9)
    ax.set_ylim(0, 10.5)
    ax.axhline(5, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Overall score comparison ─────────────────────────────────────────────────
if scored:
    fig, ax = plt.subplots(figsize=(6, 4))
    tc_ids = [r["test_id"] for r in scored]
    overall = [r["overall_score"] for r in scored]
    colors = ["#2ecc71" if s >= 7 else "#f39c12" if s >= 4 else "#e74c3c" for s in overall]
    bars = ax.barh(tc_ids, overall, color=colors)
    ax.set_xlim(0, 10)
    ax.set_xlabel("Overall Score (0–10)")
    ax.set_title("War Room Overall Score per Test Case")
    for bar, val in zip(bars, overall):
        ax.text(val + 0.1, bar.get_y() + bar.get_height() / 2, f"{val:.1f}", va="center")
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Save scores to JSON ───────────────────────────────────────────────────────
from datetime import datetime

out_path = CAPTURES_DIR.parent / "war_room_results" / f"judge_scores_{datetime.now().strftime('%Y-%m-%d_%H%M%S')}.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"Scores saved to: {out_path}")